In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
# Load the dataset
data = pd.read_csv('C:/Users/HP/Desktop/output_pose_data_012215.csv')

In [4]:
# Separate features and labels
X = data[['left_elbow_angle', 'right_elbow_angle', 'left_shoulder_angle',
          'right_shoulder_angle', 'left_knee_angle', 'right_knee_angle']]
y = data['Label']

In [22]:
# Ensure the target labels are numeric
y = pd.to_numeric(y, errors='coerce')

In [23]:
# Check for any NaN values in y and handle them
if y.isna().sum() > 0:
    print(f"Found {y.isna().sum()} NaN values in the target labels. Removing them.")
    valid_indices = y.dropna().index
    X = X.loc[valid_indices]
    y = y.dropna()

Found 938 NaN values in the target labels. Removing them.


In [24]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [6]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
# Reshape for LSTM (samples, time steps, features)
X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

In [10]:
# Save the scaler
joblib.dump(scaler, 'scaler1.pkl')

['scaler1.pkl']

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

In [15]:
# Build the LSTM model
model = Sequential()
model.add(Input(shape=(1, X_train_scaled.shape[2])))
model.add(LSTM(50, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

In [16]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [17]:
# Print the model summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                        │ (None, 1, 50)               │          11,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 1, 50)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 50)                  │          20,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 50)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 31,651 (123.64 KB)

 Trainable params: 31,651 (123.64 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
print(y_train.dtype)
print(y_test.dtype)


object
object


In [20]:
# Convert to numeric if they are not already
y_train = pd.to_numeric(y_train, errors='coerce')
y_test = pd.to_numeric(y_test, errors='coerce')

# Check if there are any NaN values and handle them if necessary
print(y_train.isna().sum())
print(y_test.isna().sum())


750
188


In [21]:
# Train the model
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.

In [25]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (0, 6)
Shape of y: (0,)


In [58]:
# Save the model
model.save('pose_verification_lstm_model.h5')

In [59]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
from tensorflow.keras.models import load_model

In [63]:
# Load the trained LSTM model and scaler
model = load_model('pose_verification_lstm_model.h5')
scaler = joblib.load('scaler1.pkl')

In [143]:
# Load the image
image_path = 'C:/Users/HP/Desktop/test_3.jpg'
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [144]:
# Initialize MediaPipe pose estimation
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, model_complexity=2)

In [145]:
# Perform pose detection
results = pose.process(image_rgb)

C:\Users\HP\anaconda3\envs\mediapipe\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [146]:
# Extract keypoints
keypoints = results.pose_landmarks.landmark if results.pose_landmarks else None

# Close the MediaPipe instance
pose.close()

In [147]:
if keypoints:
    def calculate_angle(a, b, c):
        a = np.array(a)  # First point
        b = np.array(b)  # Middle point
        c = np.array(c)  # End point
        
        radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
        angle = np.abs(radians * 180.0 / np.pi)
        
        if angle > 180.0:
            angle = 360 - angle
            
        return angle

In [148]:
try:
    left_shoulder = [keypoints[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                     keypoints[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
    left_elbow = [keypoints[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
    left_wrist = [keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

    right_shoulder = [keypoints[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,
                      keypoints[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
    right_elbow = [keypoints[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
    right_wrist = [keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
except Exception as e:
    print("Error occurred while extracting keypoints:", e)


In [149]:
try:
    left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
    left_shoulder_angle = calculate_angle(left_elbow, left_shoulder, [left_shoulder[0], left_shoulder[1] - 1])
    right_shoulder_angle = calculate_angle(right_elbow, right_shoulder, [right_shoulder[0], right_shoulder[1] - 1])
    
    left_knee = [keypoints[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                 keypoints[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
    left_hip = [keypoints[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                keypoints[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_ankle = [keypoints[mp_pose.PoseLandmark.LEFT_ANKLE.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]

    right_knee = [keypoints[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,
                  keypoints[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
    right_hip = [keypoints[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
                 keypoints[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
    right_ankle = [keypoints[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]

    left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
    right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)
except Exception as e:
    print("Error occurred while calculating angles:", e)


In [150]:
try:
    angles_df = pd.DataFrame([{
        'left_elbow_angle': left_elbow_angle,
        'right_elbow_angle': right_elbow_angle,
        'left_shoulder_angle': left_shoulder_angle,
        'right_shoulder_angle': right_shoulder_angle,
        'left_knee_angle': left_knee_angle,
        'right_knee_angle': right_knee_angle
    }])
except Exception as e:
    print("Error occurred while creating DataFrame for angles:", e)


In [151]:
try:
    angles_df = angles_df[['left_elbow_angle', 'right_elbow_angle', 'left_shoulder_angle',
                           'right_shoulder_angle', 'left_knee_angle', 'right_knee_angle']]
except Exception as e:
    print("Error occurred while ensuring correct column order in DataFrame:", e)

In [152]:
try:
    angles_scaled = scaler.transform(angles_df)
except Exception as e:
    print("Error occurred while scaling angles using the fitted scaler:", e)


In [153]:
try:
    angles_scaled = angles_scaled.reshape((1, 1, angles_scaled.shape[1]))
except Exception as e:
    print("Error occurred while reshaping angles for LSTM model:", e)


In [154]:
if keypoints is not None and model is not None and scaler is not None:
    try:
        pose_correctness = model.predict(angles_scaled)
        print(f"Pose Correctness: {'Correct' if pose_correctness[0][0] > 0.5 else 'Incorrect'}")
    except Exception as e:
        print(f"Error in calculating angles or prediction: {e}")
else:
    print("No keypoints detected")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
Pose Correctness: Correct
